# Benchmark: All Models on NASA Ames Multi-Temperature Dataset
## TE-Q-Transformer Research Framework

**Manuscript Title:** *TE-Q-Transformer: A Temperature-Embedded Quantum Framework for Battery State-of-Health Estimation*

This notebook provides the unified training and evaluation benchmark for:
1. **TE-Q-Transformer (Proposed)**: Physics-guided Arrhenius encoding + 4-qubit simulated variational quantum circuit + Conv1D-Transformer backbone.
2. **10 Baseline Architectures**:
   - Recurrent: **LSTM**, **GRU**
   - Convolutional: **CNN1D**, **TCN** (Causal Dilated)
   - Linear Decomposition: **DLinear**
   - Self-Attention & Transformers: **Transformer**, **PatchTST**, **iTransformer**
   - Quantum-Classical Hybrids: **QLSTM** (Gate-level VQC), **QNN-GRU** (Variational feature map + GRU)

---

### Evaluation Protocol:
- **Sequence Length:** Exactly 512 time steps ($V$, $I$, $T_C$, $t_{\text{norm}}$).
- **Training Pool (660 cycles):** `B0005`, `B0006`, `B0007` (24°C), `B0029`, `B0030`, `B0031` (4°C), and first 70% of `B0053` (44°C).
- **Held-Out Test Cells (187 cycles):** Unseen cells `B0018` (24°C), `B0032` (4°C), and final 30% of `B0053` (44°C).
- **Zero-Leakage Scaling:** MinMaxScaler fitted exclusively on the training pool for channels (0, 1, 3). Temperature (channel 2) is left in unscaled Celsius for Arrhenius physics semantics.

In [ ]:
# ==============================================================================
# 0. SETUP ENVIRONMENT AND REPOSITORY PATHS
# ==============================================================================
import os
import sys
from pathlib import Path

MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / REPO_NAME,
    Path("/kaggle/working"),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")
DATA_ROOT = REPO_ROOT / "datasets"
MODEL_ROOT = REPO_ROOT / "models"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware device: {DEVICE}")

## 1. Load Dataset
Loading processed NASA Ames cycle sequences ($N \times 512 \times 4$) and creating PyTorch DataLoaders.

In [ ]:
from src.data.nasa_loader import get_nasa_dataloaders

nasa_dir = DATA_ROOT / "NASA" / "processed"
train_loader, test_loaders, scaler = get_nasa_dataloaders(nasa_dir, batch_size=8)

print(f"[Dataset] Train loader: {len(train_loader)} batches ({len(train_loader.dataset)} cycles)")
for cell_id, loader in test_loaders.items():
    print(f"  - Test cell {cell_id:<12}: {len(loader.dataset)} cycles ({len(loader)} batches)")

## 2. Model Registry
Instantiating proposed TE-Q-Transformer and all 10 baseline architectures to inspect parameter footprints.

In [ ]:
from models.proposed import TEQTransformer, rich_entangler_config
from models.baselines import get_baseline_model, list_baselines

def get_model(name: str):
    if name == "TE-Q-Transformer":
        return TEQTransformer(rich_entangler_config())
    return get_baseline_model(name)

all_names = ["TE-Q-Transformer"] + list_baselines()

print("=" * 65)
print(f"{'Model Architecture':<30} | {'Trainable Parameters':<20}")
print("=" * 65)
for name in all_names:
    m = get_model(name)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:<30} | {n_params:<20,d}")
print("=" * 65)

## 3. Training & Evaluation Pipeline
Modular functions for model optimization, validation, and multi-metric computation.

In [ ]:
from src.eval.metrics import calculate_metrics, calculate_macro_metrics

def train_epoch(model: nn.Module, loader: DataLoader, optimizer: optim.Optimizer, criterion: nn.Module, device: torch.device) -> float:
    model.train()
    total_loss = 0.0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        pred = model(bx)
        loss = criterion(pred, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
    return total_loss / len(loader.dataset)

def evaluate_model(model: nn.Module, test_loaders: dict, device: torch.device) -> dict:
    model.eval()
    per_cell = {}
    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            preds, actuals = [], []
            for bx, by in loader:
                bx = bx.to(device)
                pred = model(bx)
                preds.append(pred.cpu().numpy())
                actuals.append(by.numpy())
            y_pred = np.concatenate(preds)
            y_true = np.concatenate(actuals)
            per_cell[cell_id] = {
                "metrics": calculate_metrics(y_true, y_pred),
                "preds": y_pred,
                "actuals": y_true,
            }
    macro = calculate_macro_metrics([res["metrics"] for res in per_cell.values()])
    return {"per_cell": per_cell, "macro": macro}

def run_experiment(model_name: str, epochs: int = 10, lr: float = 1e-3, device: torch.device = DEVICE) -> dict:
    print(f"\n>>> Starting Experiment: {model_name} ({epochs} epochs, lr={lr}) <<<")
    model = get_model(model_name).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    for ep in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, criterion, device)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            print(f"Epoch {ep:3d}/{epochs:3d} | Train MSE Loss: {loss:.6f}")
    
    eval_res = evaluate_model(model, test_loaders, device)
    print(f"\nEvaluation Results for {model_name}:")
    for cell_id, cdata in eval_res["per_cell"].items():
        m = cdata["metrics"]
        print(f"  - Cell {cell_id:<12}: RMSE={m['RMSE']:.5f} | MAE={m['MAE']:.5f} | R2={m['R2']:.5f}")
    macro = eval_res["macro"]
    print(f"  >> Macro Average   : RMSE={macro['RMSE']:.5f} | MAE={macro['MAE']:.5f} | R2={macro['R2']:.5f}")
    return {"model": model, "results": eval_res}

## 4. Run Example Model Benchmark
Train and evaluate any model architecture directly.

In [ ]:
# Example: Train and evaluate CNN1D (or replace with 'TE-Q-Transformer', 'LSTM', etc.)
exp = run_experiment("CNN1D", epochs=5, lr=1e-3, device=DEVICE)

## 5. SOH Trajectory Visualization
Plotting ground truth vs predicted State-of-Health degradation curves for evaluated test cells.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (cell_id, cdata) in zip(axes, exp["results"]["per_cell"].items()):
    y_true = cdata["actuals"]
    y_pred = cdata["preds"]
    cycles = np.arange(len(y_true))
    ax.plot(cycles, y_true, 'k-', lw=2, label="Actual SOH")
    ax.plot(cycles, y_pred, 'r--', lw=2, label=f"Predicted")
    m = cdata["metrics"]
    ax.set_title(f"Cell {cell_id} (RMSE: {m['RMSE']:.4f}, R²: {m['R2']:.3f})")
    ax.set_xlabel("Cycle Index")
    ax.set_ylabel("SOH")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()